In [ ]:
# Cell 1 — self-contained bootstrap
from google.colab import userdata
import torch, os, sys

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_EMAIL = "evenjlinekka@gmail.com"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Evenjlin/deep-space-interference-ml.git"

if not os.path.exists('/content/deep-space-interference-ml'):
    !git clone {REPO_URL} /content/deep-space-interference-ml
%cd /content/deep-space-interference-ml
!git config --global user.email "{GITHUB_EMAIL}"
!git config --global user.name "Evenjlin"
!git remote set-url origin {REPO_URL}
!git pull

!pip install -r requirements.txt -q
sys.path.insert(0, os.getcwd())

In [ ]:
# Cell 2 — data pipeline WITH metadata (bits, fd, phi_m, k_shift) for the decision-aware loss
import numpy as np
import torch
from src.channel import SignalConfig, mix_signal
from src.model import MitigationAutoencoder, complex_to_channels
from src.evaluate import matched_filter_ber, theoretical_ber
from src.train import decision_aware_loss, train_mitigation_ae_decision_aware

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = SignalConfig(fd_max=0.0)
rng = np.random.default_rng(42)

N_SYMBOLS = 1024  # BASE-PAPER FACT, same as Step 8b/c for direct comparability
SNR_RANGE_TRAIN = (-15, 30)
SIR_RANGE_TRAIN = (-10, 30)
INTERFERENCE_TYPE = "tone"  # start here -- already have Step 8b's MSE-only baseline to compare against

def make_mitigation_pair_with_meta(cfg, n_symbols, snr_range, sir_range, interference_type, rng):
    d = mix_signal(cfg, n_symbols, rng, interference_type=interference_type,
                    contamination=0.5, sir_db_range=sir_range, snr_db_range=snr_range, apply_shift=False)
    X = complex_to_channels(d["x"]).astype(np.float32)
    Y = complex_to_channels(d["s"]).astype(np.float32)
    return X, Y, d["bits"], d["fd"], d["phi_m"], d["k_shift"]

def build_dataset_with_meta(n):
    X, Y, bits, fd, phi_m, k_shift = [], [], [], [], [], []
    for _ in range(n):
        x, y, b, f, p, k = make_mitigation_pair_with_meta(cfg, N_SYMBOLS, SNR_RANGE_TRAIN, SIR_RANGE_TRAIN, INTERFERENCE_TYPE, rng)
        X.append(x); Y.append(y); bits.append(b); fd.append(f); phi_m.append(p); k_shift.append(k)
    return (np.stack(X), np.stack(Y), np.stack(bits).astype(np.int64),
            np.array(fd, dtype=np.float32), np.array(phi_m, dtype=np.float32), np.array(k_shift, dtype=np.int64))

M_TRAIN, M_VAL = 1500, 400
X_train, Y_train, bits_train, fd_train, phi_train, k_train = build_dataset_with_meta(M_TRAIN)
X_val, Y_val, bits_val, fd_val, phi_val, k_val = build_dataset_with_meta(M_VAL)
print("X_train:", X_train.shape, " bits_train:", bits_train.shape, " fd range:", fd_train.min(), fd_train.max())

In [ ]:
# Cell 3 — DEBUG LADDER for the new loss function (critical: this is genuinely new, untested code)
model = MitigationAutoencoder(n_stages=2, n_hidden=32).to(device)

tiny_x = torch.tensor(X_train[:8], device=device)
tiny_y = torch.tensor(Y_train[:8], device=device)
tiny_bits = torch.tensor(bits_train[:8], device=device, dtype=torch.long)
tiny_fd = torch.tensor(fd_train[:8], device=device, dtype=torch.float32)
tiny_phi = torch.tensor(phi_train[:8], device=device, dtype=torch.float32)
tiny_k = torch.tensor(k_train[:8], device=device, dtype=torch.long)

out = model(tiny_x)
print("TEST 1: output shape", out.shape)
assert out.shape == tiny_x.shape

loss, mse_val, dec_val = decision_aware_loss(out, tiny_y, tiny_bits, tiny_fd, tiny_phi, tiny_k, cfg, N_SYMBOLS)
print(f"TEST 2: loss computed OK -- total={loss.item():.4f}  mse={mse_val:.4f}  decision={dec_val:.4f}")

opt = torch.optim.Adam(model.parameters(), lr=3e-5)
loss.backward()
print("TEST 3: backward() succeeded -- gradients flow through the de-rotation math")
opt.step(); opt.zero_grad()

# TEST 4: a few steps, confirm total loss decreases
small_x = torch.tensor(X_train[:200], device=device)
small_y = torch.tensor(Y_train[:200], device=device)
small_bits = torch.tensor(bits_train[:200], device=device, dtype=torch.long)
small_fd = torch.tensor(fd_train[:200], device=device, dtype=torch.float32)
small_phi = torch.tensor(phi_train[:200], device=device, dtype=torch.float32)
small_k = torch.tensor(k_train[:200], device=device, dtype=torch.long)

losses = []
for _ in range(20):
    opt.zero_grad()
    out = model(small_x)
    loss, mse_val, dec_val = decision_aware_loss(out, small_y, small_bits, small_fd, small_phi, small_k, cfg, N_SYMBOLS)
    loss.backward(); opt.step()
    losses.append((loss.item(), mse_val, dec_val))

print("TEST 4 losses (total, mse, decision):")
for t, m, d in losses:
    print(f"  total={t:.4f}  mse={m:.4f}  decision={d:.4f}")
assert losses[-1][0] < losses[0][0], "Total loss did not decrease -- STOP, don't proceed"
print("Debug ladder PASSED.")